# W04 — Leakage-safe baseline action score: Content-Day Decline Risk

**Lane:** Content-Day Performance / Decline Risk.  
**Prediction cutoff:** 2026-03-16. **Feature window:** 2026-03-01 through 2026-03-15. **Outcome window:** 2026-03-16 through 2026-03-31.  
**Grain:** one row per client × content item.

## Important correction
The first draft ranked pages using the click decline measured in the outcome window. That leaks the answer into a predictive baseline. This corrected version uses only information available by March 15 to rank pages. Late-March clicks define the evaluation outcome and support the top-10 review; they do not enter the score.

## Rule idea
Rank eligible pages by early search visibility using the fixed score log1p(early_impressions). This is a transparent prioritization heuristic, not a calibrated probability or causal claim.

In [4]:
# Colab setup. Store HF_TOKEN in Colab Secrets; never paste credentials into the notebook.
!pip -q install duckdb pandas numpy

from pathlib import Path
import json
import subprocess
import duckdb
import numpy as np
import pandas as pd
from IPython.display import display
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("Add your Hugging Face read token as a Colab Secret named HF_TOKEN.")

# Find the repository root locally; clone the public repo when opened directly from GitHub in Colab.
REPO_URL = "https://github.com/yaseen20051/FlyRank_Internship.git"
candidate_roots = [Path.cwd(), Path("/content/FlyRank_Internship")]
REPO_ROOT = next((p for p in candidate_roots if (p / "work").exists()), None)
if REPO_ROOT is None:
    clone_path = Path("/content/FlyRank_Internship")
    if not clone_path.exists():
        subprocess.run(["git", "clone", REPO_URL, str(clone_path)], check=True)
    REPO_ROOT = clone_path

OUTPUT_DIR = REPO_ROOT / "work" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BASE = "hf://datasets/FlyRank/internship-warehouse"
DEV_MONTH = "2026-03"
DECISION_DATE = "2026-04-01"
EARLY_START, EARLY_END = "2026-03-01", "2026-03-15"
LATE_START, LATE_END = "2026-03-16", "2026-03-31"
MONTH_GLOB = f"{BASE}/fact_content_daily_performance/month={DEV_MONTH}/*.parquet"

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("SET s3_url_style='path';")
con.execute(f"CREATE OR REPLACE SECRET hf_token (TYPE huggingface, TOKEN '{HF_TOKEN}');")

print("Repository:", REPO_ROOT)
print("Feature window:", EARLY_START, "through", LATE_END)
print("Decision date:", DECISION_DATE)
print("Output directory:", OUTPUT_DIR)

Repository: /content/FlyRank_Internship
Feature window: 2026-03-01 through 2026-03-31
Decision date: 2026-04-01
Output directory: /content/FlyRank_Internship/work/outputs


## 1) Confirm the source, grain, and usable observations

Use only rows where gsc_data_available IS TRUE. A false or null availability flag is not evidence of zero traffic. Treat average position 0 as missing, not as a real rank. The W03 notebook already verified the March fact-table grain as client × content × report date and found no duplicate-grain rows; this cell repeats the essential checks for this run.

In [5]:
fact_schema = con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{MONTH_GLOB}')"
).fetchdf()
content_schema = con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{BASE}/dim_content.parquet')"
).fetchdf()

print("Fact table schema:")
display(fact_schema)
print("Content dimension schema:")
display(content_schema)

required_fact = {
    "report_date", "client_hash_id", "content_hash_id",
    "gsc_data_available", "gsc_impressions", "gsc_clicks", "gsc_avg_position"
}
required_dim = {"client_hash_id", "content_hash_id", "content_created_date", "content_updated_date"}
assert required_fact.issubset(set(fact_schema["column_name"]))
assert required_dim.issubset(set(content_schema["column_name"]))

dimension_duplicate_keys = con.execute(f"""
    SELECT COUNT(*) FROM (
        SELECT client_hash_id, content_hash_id
        FROM read_parquet('{BASE}/dim_content.parquet')
        GROUP BY 1, 2
        HAVING COUNT(*) > 1
    )
""").fetchone()[0]
print("Duplicate content-dimension client/content keys:", dimension_duplicate_keys)
assert dimension_duplicate_keys == 0, "Content dimension join would multiply page rows."

month_check = con.execute(f"""
    SELECT
        COUNT(*) AS fact_rows,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_items,
        MIN(report_date) AS first_report_date,
        MAX(report_date) AS last_report_date,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows
    FROM read_parquet('{MONTH_GLOB}')
""").fetchdf()
display(month_check)

duplicate_check = con.execute(f"""
    SELECT COUNT(*) AS duplicate_grains
    FROM (
        SELECT client_hash_id, content_hash_id, report_date
        FROM read_parquet('{MONTH_GLOB}')
        GROUP BY 1, 2, 3
        HAVING COUNT(*) > 1
    )
""").fetchone()[0]
print("Duplicate client-content-date grains:", duplicate_check)
assert duplicate_check == 0, "Unexpected duplicate daily grain; stop and investigate."

Fact table schema:


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


Content dimension schema:


,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate content-dimension client/content keys: 0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,fact_rows,clients,content_items,first_report_date,last_report_date,gsc_available_rows
0,9841378,55,331437,2026-03-01,2026-03-31,3611061.0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate client-content-date grains: 0


## 1A) Build one page-level, point-in-time March snapshot

The first half (March 1–15) is the earlier reference period; the second half (March 16–31) is the latest observed period at the March 31 decision point. Clicks are normalized by the number of measured GSC days in each half, so pages with different observed-day counts are not compared as raw totals.

Freshness is calculated from content_updated_date as of March 31. Missing or future-dated update values remain unknown; they are never silently interpreted as stale.

In [6]:
con.execute(f"""
    CREATE OR REPLACE VIEW dim_content AS
    SELECT * FROM read_parquet('{BASE}/dim_content.parquet')
""")

page_sql = f"""
WITH daily AS (
    SELECT
        CAST(report_date AS DATE) AS report_date,
        client_hash_id,
        content_hash_id,
        COALESCE(gsc_impressions, 0)::DOUBLE AS impressions,
        COALESCE(gsc_clicks, 0)::DOUBLE AS clicks
    FROM read_parquet('{MONTH_GLOB}')
    WHERE CAST(report_date AS DATE)
          BETWEEN DATE '{EARLY_START}' AND DATE '{LATE_END}'
      AND gsc_data_available IS TRUE
),
page_periods AS (
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(*) FILTER (
            WHERE report_date BETWEEN DATE '{EARLY_START}' AND DATE '{EARLY_END}'
        ) AS early_observations,
        SUM(CASE WHEN report_date BETWEEN DATE '{EARLY_START}' AND DATE '{EARLY_END}'
                 THEN impressions ELSE 0 END) AS early_impressions,
        SUM(CASE WHEN report_date BETWEEN DATE '{EARLY_START}' AND DATE '{EARLY_END}'
                 THEN clicks ELSE 0 END) AS early_clicks,
        COUNT(*) FILTER (
            WHERE report_date BETWEEN DATE '{LATE_START}' AND DATE '{LATE_END}'
        ) AS late_observations,
        SUM(CASE WHEN report_date BETWEEN DATE '{LATE_START}' AND DATE '{LATE_END}'
                 THEN impressions ELSE 0 END) AS late_impressions,
        SUM(CASE WHEN report_date BETWEEN DATE '{LATE_START}' AND DATE '{LATE_END}'
                 THEN clicks ELSE 0 END) AS late_clicks
    FROM daily
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    p.client_hash_id,
    p.content_hash_id,
    p.early_observations,
    p.early_impressions,
    p.early_clicks,
    p.late_observations,
    p.late_impressions,
    p.late_clicks,
    d.content_created_date,
    d.content_updated_date
FROM page_periods p
LEFT JOIN dim_content d
  ON d.client_hash_id = p.client_hash_id
 AND d.content_hash_id = p.content_hash_id
WHERE p.early_observations >= 10
  AND p.late_observations >= 10
"""

pages = con.execute(page_sql).fetchdf()
for col in ["early_impressions", "early_clicks", "late_impressions", "late_clicks",
            "early_observations", "late_observations"]:
    pages[col] = pd.to_numeric(pages[col], errors="coerce").fillna(0)

pages["early_clicks_per_day"] = pages["early_clicks"] / pages["early_observations"].replace(0, np.nan)
pages["late_clicks_per_day"] = pages["late_clicks"] / pages["late_observations"].replace(0, np.nan)
pages["click_change_pct"] = (
    (pages["late_clicks_per_day"] / pages["early_clicks_per_day"]) - 1
) * 100
pages.loc[pages["early_clicks_per_day"] <= 0, "click_change_pct"] = np.nan

update_dates = pd.to_datetime(pages["content_updated_date"], errors="coerce")
decision_ts = pd.Timestamp(DECISION_DATE)
pages["days_since_update"] = (decision_ts - update_dates).dt.days
pages.loc[pages["days_since_update"] < 0, "days_since_update"] = np.nan

print(f"Page rows with >=10 measured GSC observations in each half: {len(pages):,}")
print("Early-period impressions total:", f"{pages['early_impressions'].sum():,.0f}")
print("Late-period impressions total:", f"{pages['late_impressions'].sum():,.0f}")
display(pages.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Page rows with >=10 measured GSC observations in each half: 96,676
Early-period impressions total: 124,950,745
Late-period impressions total: 141,799,594


,client_hash_id,content_hash_id,early_observations,early_impressions,early_clicks,late_observations,late_impressions,late_clicks,content_created_date,content_updated_date,early_clicks_per_day,late_clicks_per_day,click_change_pct,days_since_update
0,client_62f4a7e64f5e0096,content_c03ecafd4c999f15,15,4974.0,9.0,16,5875.0,13.0,2026-02-12,2026-06-29,0.600000,0.8125,35.416667,NaN
1,client_62f4a7e64f5e0096,content_e689bc511192751a,15,35.0,0.0,12,26.0,0.0,2026-02-12,2026-06-29,0.000000,0.0000,NaN,NaN
2,client_62f4a7e64f5e0096,content_7dbc094b799e05a4,15,398.0,1.0,16,307.0,0.0,2026-02-12,2026-06-29,0.066667,0.0000,-100.000000,NaN
3,client_62f4a7e64f5e0096,content_df22bda1218f13ff,15,670.0,0.0,16,1429.0,1.0,2026-02-12,2026-06-29,0.000000,0.0625,NaN,NaN
4,client_62f4a7e64f5e0096,content_aa184c1b4ea518e5,15,133.0,0.0,15,155.0,0.0,2026-02-12,2026-06-29,0.000000,0.0000,NaN,NaN


## 1B) Signal check 1 — search volume (quick-win flag family)

**Claim:** pages with more early-period impressions carry more observed click opportunity. Volume is one of the signals behind FlyRank's quick-win logic.

The table prints the number of pages (n) in every bucket. Verdicts use only buckets with at least 50 pages; the sample-size floor prevents tiny buckets from driving the conclusion.

In [7]:
volume_edges = [-0.001, 99, 499, 1999, np.inf]
volume_labels = ["0–99", "100–499", "500–1,999", "2,000+"]
pages["volume_bucket"] = pd.cut(
    pages["early_impressions"], bins=volume_edges, labels=volume_labels,
    include_lowest=True, ordered=True
)
pages["volume_order"] = pages["volume_bucket"].cat.codes + 1

volume_rows = []
for bucket, group in pages.groupby("volume_bucket", observed=False):
    if pd.isna(bucket):
        continue
    total_impressions = group["early_impressions"].sum()
    total_clicks = group["early_clicks"].sum()
    volume_rows.append({
        "volume_bucket": str(bucket),
        "bucket_order": int(group["volume_order"].iloc[0]) if len(group) else np.nan,
        "n": int(len(group)),
        "median_early_impressions": float(group["early_impressions"].median()) if len(group) else np.nan,
        "median_early_clicks_per_day": float(group["early_clicks_per_day"].median()) if len(group) else np.nan,
        "weighted_ctr_pct": (100 * total_clicks / total_impressions) if total_impressions > 0 else np.nan,
        "share_click_decline_20pct": float((group["click_change_pct"] <= -20).mean()) if len(group) else np.nan,
    })

volume_table = pd.DataFrame(volume_rows)
display(volume_table)

def verdict_from_bucket_table(table, order_col, metric_col, min_n=50):
    usable = table.loc[table["n"] >= min_n, [order_col, metric_col]].dropna()
    if len(usable) < 3:
        return "MIXED", np.nan, f"Fewer than three buckets meet the n≥{min_n} floor."
    rho = usable[order_col].corr(usable[metric_col], method="spearman")
    if pd.isna(rho):
        return "MIXED", np.nan, "The bucket trend could not be estimated."
    if rho >= 0.30:
        return "CONFIRMED", float(rho), "Higher-volume buckets have higher median early clicks per measured day."
    if rho <= -0.30:
        return "OPPOSITE", float(rho), "Higher-volume buckets have lower median early clicks per measured day."
    if abs(rho) < 0.10:
        return "FALSE", float(rho), "No useful monotonic volume-to-click pattern is visible in these buckets."
    return "MIXED", float(rho), "The relationship is weak or inconsistent across the usable buckets."

volume_verdict, volume_rho, volume_note = verdict_from_bucket_table(
    volume_table, "bucket_order", "median_early_clicks_per_day"
)
print("VOLUME VERDICT:", volume_verdict)
print("Spearman across bucket summaries:", volume_rho)
print(volume_note)

,volume_bucket,bucket_order,n,median_early_impressions,median_early_clicks_per_day,weighted_ctr_pct,share_click_decline_20pct
0,0–99,1,23137,52.0,0.000000,0.240468,0.083200
1,100–499,2,32648,226.0,0.000000,0.246705,0.218206
2,"500–1,999",3,25676,924.0,0.133333,0.315339,0.378213
3,"2,000+",4,15215,3805.0,0.600000,0.298232,0.440815


VOLUME VERDICT: CONFIRMED
Spearman across bucket summaries: 0.9486832980505139
Higher-volume buckets have higher median early clicks per measured day.


## 1C) Signal check 2 — content freshness (refresh flag family)

**Claim:** pages not updated for longer may be more likely to show a meaningful observed click-rate drop between the two March halves. Freshness is a signal behind FlyRank's refresh flags.

Unknown update dates stay in a separate bucket and do not count as stale. The rule will use a freshness multiplier only when this check returns CONFIRMED; an OPPOSITE, MIXED, or FALSE result will leave freshness out of the score.

In [8]:
fresh_edges = [-0.001, 30, 90, 180, np.inf]
fresh_labels = ["0–30 days", "31–90 days", "91–180 days", "181+ days"]
pages["freshness_bucket"] = pd.cut(
    pages["days_since_update"], bins=fresh_edges, labels=fresh_labels,
    include_lowest=True, ordered=True
)
pages["freshness_order"] = pages["freshness_bucket"].cat.codes + 1
pages.loc[pages["days_since_update"].isna(), "freshness_order"] = np.nan
pages["freshness_bucket_display"] = pages["freshness_bucket"].astype("object")
pages.loc[pages["days_since_update"].isna(), "freshness_bucket_display"] = "unknown"

fresh_rows = []
for bucket in ["0–30 days", "31–90 days", "91–180 days", "181+ days", "unknown"]:
    group = pages.loc[pages["freshness_bucket_display"] == bucket]
    if group.empty:
        continue
    fresh_rows.append({
        "freshness_bucket": bucket,
        "bucket_order": (
            float(group["freshness_order"].dropna().iloc[0])
            if group["freshness_order"].notna().any() else np.nan
        ),
        "n": int(len(group)),
        "median_days_since_update": (
            float(group["days_since_update"].median())
            if group["days_since_update"].notna().any() else np.nan
        ),
        "median_click_change_pct": float(group["click_change_pct"].median()),
        "share_click_decline_20pct": float((group["click_change_pct"] <= -20).mean()),
    })

freshness_table = pd.DataFrame(fresh_rows)
display(freshness_table)

freshness_ordered = freshness_table.dropna(subset=["bucket_order"]).copy()
freshness_verdict, freshness_rho, freshness_note = verdict_from_bucket_table(
    freshness_ordered, "bucket_order", "share_click_decline_20pct"
)
print("FRESHNESS VERDICT:", freshness_verdict)
print("Spearman across freshness buckets:", freshness_rho)
print(freshness_note)
unknown_n = int(pages["days_since_update"].isna().sum())
print(f"Unknown/future-dated update timestamps (kept neutral): n={unknown_n:,}")

,freshness_bucket,bucket_order,n,median_days_since_update,median_click_change_pct,share_click_decline_20pct
0,0–30 days,1.0,87,23.0,-100.000000,0.206897
1,31–90 days,2.0,18768,35.0,-53.125000,0.250799
2,91–180 days,3.0,153,125.0,-37.500000,0.228758
3,181+ days,4.0,16,235.0,-43.750000,0.125000
4,unknown,NaN,77652,NaN,-27.083333,0.266638


FRESHNESS VERDICT: CONFIRMED
Spearman across freshness buckets: 0.5
Higher-volume buckets have higher median early clicks per measured day.
Unknown/future-dated update timestamps (kept neutral): n=77,652


## 2) Encode ONE leakage-safe rule

Eligibility requires at least 100 early impressions, at least 5 early clicks, and at least 10 measured GSC observations in each half. The later window is used only to define the outcome label.

**Frozen baseline score:** log1p(early_impressions). The score is intentionally simple and uses only information available by the prediction cutoff. The queue contains every eligible page, ranked using early-period impressions; no late-window field enters the score.

In [9]:
MIN_EARLY_IMPRESSIONS = 100
MIN_EARLY_CLICKS = 5
MAX_CLICK_CHANGE_PCT = -20.0

eligible = pages.loc[
    (pages["early_impressions"] >= MIN_EARLY_IMPRESSIONS)
    & (pages["early_clicks"] >= MIN_EARLY_CLICKS)
    & pages["click_change_pct"].notna()
].copy()

# Outcome label is later-period evidence; it is never used in the ranking score.
eligible["decline_label"] = (eligible["click_change_pct"] <= MAX_CLICK_CHANGE_PCT).astype(int)
eligible["freshness_multiplier"] = (
    1 + eligible["days_since_update"].clip(lower=0, upper=365).fillna(0) / 365
)
eligible["baseline_action_score"] = (
    np.log1p(eligible["early_impressions"])
)
eligible["reason_code"] = "visible_and_due_for_review"
eligible["action_label"] = "review_search_performance"

queue = eligible.sort_values(
    ["baseline_action_score", "early_impressions"], ascending=[False, False]
).reset_index(drop=True)
queue.insert(0, "baseline_rank", np.arange(1, len(queue) + 1))

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    k_eff = min(k, len(order))
    return float(np.asarray(labels)[order[:k_eff]].mean()) if k_eff else np.nan

base_rate = float(queue["decline_label"].mean()) if len(queue) else np.nan
baseline_metrics = {
    f"precision_at_{k}": precision_at_k(queue["baseline_action_score"], queue["decline_label"], k)
    for k in (10, 20, 50)
}
print("Eligible pages:", f"{len(queue):,}")
print("Decline-proxy base rate:", f"{base_rate:.3f}")
print("Early-only baseline:", baseline_metrics)

OUTPUT_CSV = OUTPUT_DIR / "baseline_action_score.csv"
output_columns = [
    "baseline_rank", "client_hash_id", "content_hash_id", "baseline_action_score",
    "reason_code", "action_label", "early_impressions", "early_clicks",
    "early_observations", "early_clicks_per_day", "late_impressions", "late_clicks",
    "late_observations", "late_clicks_per_day", "click_change_pct", "decline_label",
    "days_since_update",
]
queue[output_columns].to_csv(OUTPUT_CSV, index=False)
display(queue[output_columns].head(10))

Eligible pages: 16,833
Decline-proxy base rate: 0.521
Early-only baseline: {'precision_at_10': 0.4, 'precision_at_20': 0.3, 'precision_at_50': 0.36}


,baseline_rank,client_hash_id,content_hash_id,baseline_action_score,reason_code,action_label,early_impressions,early_clicks,early_observations,early_clicks_per_day,late_impressions,late_clicks,late_observations,late_clicks_per_day,click_change_pct,decline_label,days_since_update
0,1,client_e547b89c05043229,content_eadb33b5df496f4a,11.992731,visible_and_due_for_review,review_search_performance,161575.0,2395.0,13,184.230769,455549.0,3273.0,16,204.5625,11.036013,0,NaN
1,2,client_23a62021009f63c4,content_e8a52cf3d5988c07,11.871816,visible_and_due_for_review,review_search_performance,143173.0,353.0,15,23.533333,101758.0,316.0,16,19.7500,-16.076487,0,NaN
2,3,client_e547b89c05043229,content_ec2e0346994fb5a5,11.796690,visible_and_due_for_review,review_search_performance,132811.0,843.0,13,64.846154,112465.0,637.0,16,39.8125,-38.604686,1,NaN
3,4,client_23a62021009f63c4,content_36e53e9c707674fc,11.607417,visible_and_due_for_review,review_search_performance,109909.0,115.0,15,7.666667,84670.0,127.0,16,7.9375,3.532609,0,NaN
4,5,client_62f4a7e64f5e0096,content_7172a7fad43f0998,11.596016,visible_and_due_for_review,review_search_performance,108663.0,493.0,15,32.866667,97204.0,369.0,16,23.0625,-29.830122,1,NaN
5,6,client_62f4a7e64f5e0096,content_b99ea6861864dea5,11.423821,visible_and_due_for_review,review_search_performance,91474.0,185.0,15,12.333333,102863.0,176.0,16,11.0000,-10.810811,0,NaN
6,7,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,11.381255,visible_and_due_for_review,review_search_performance,87662.0,968.0,15,64.533333,117383.0,1478.0,16,92.3750,43.143079,0,NaN
7,8,client_62f4a7e64f5e0096,content_7c6373141eae744a,11.372064,visible_and_due_for_review,review_search_performance,86860.0,51.0,15,3.400000,45733.0,32.0,16,2.0000,-41.176471,1,NaN
8,9,client_23a62021009f63c4,content_3df3f32f3fd58dea,11.339072,visible_and_due_for_review,review_search_performance,84041.0,120.0,15,8.000000,56115.0,77.0,16,4.8125,-39.843750,1,NaN
9,10,client_62f4a7e64f5e0096,content_acbcc847f8996314,11.335185,visible_and_due_for_review,review_search_performance,83715.0,133.0,15,8.866667,87093.0,129.0,16,8.0625,-9.069549,0,NaN


## 3) Top-10 review — action, why it is here, and what would make it wrong

This is a human review, not an automatic endorsement. Each recommendation gets a row-specific explanation and a concrete failure condition. If fewer than ten pages pass the fixed rule, review every passing page and report the shortfall rather than padding the list with non-candidates.

In [10]:
top10 = queue.head(10).copy()
print(f"Top-10 review rows: {len(top10)} of the {len(queue)} actionable candidates")

def wrong_if(row):
    caveats = []
    if row["early_impressions"] < 500:
        caveats.append("the earlier impression base is modest, so normal search noise may explain the change")
    if row["early_clicks"] < 10:
        caveats.append("the earlier click count is small, making the percentage drop unstable")
    if row["early_observations"] < 14 or row["late_observations"] < 15:
        caveats.append("not all calendar days were measured, so missing coverage may distort the per-day comparison")
    if pd.isna(row["days_since_update"]):
        caveats.append("the update timestamp is unknown, so freshness cannot support this case")
    caveats.append("query mix, seasonality, SERP layout changes, or tracking changes could explain the drop without a content problem")
    return "; ".join(caveats)

review_lines = []
for _, row in top10.iterrows():
    why = (
        f"{row['early_impressions']:,.0f} impressions and {row['early_clicks']:,.0f} clicks "
        f"in the first half; clicks per measured day changed by {row['click_change_pct']:.1f}% "
        f"in the second half"
    )
    line = (
        f"{int(row['baseline_rank'])}. {row['content_hash_id']} — "
        f"Action: {row['action_label']}; why: {why}; "
        f"what could make it wrong: {wrong_if(row)}."
    )
    review_lines.append(line)
    print(line)

if len(top10) < 10:
    print(f"NOTE: only {len(top10)} rows met the fixed rule. Do not invent or pad recommendations.")

Top-10 review rows: 10 of the 16833 actionable candidates
1. content_eadb33b5df496f4a — Action: review_search_performance; why: 161,575 impressions and 2,395 clicks in the first half; clicks per measured day changed by 11.0% in the second half; what could make it wrong: not all calendar days were measured, so missing coverage may distort the per-day comparison; the update timestamp is unknown, so freshness cannot support this case; query mix, seasonality, SERP layout changes, or tracking changes could explain the drop without a content problem.
2. content_e8a52cf3d5988c07 — Action: review_search_performance; why: 143,173 impressions and 353 clicks in the first half; clicks per measured day changed by -16.1% in the second half; what could make it wrong: the update timestamp is unknown, so freshness cannot support this case; query mix, seasonality, SERP layout changes, or tracking changes could explain the drop without a content problem.
3. content_ec2e0346994fb5a5 — Action: review_searc

## 4) Weak picks — inspect the least-supported recommendations

The weakest recommendations are the top-10 rows with the smallest supporting evidence (lowest early impressions/click counts). A high score does not erase a small denominator; these are the first cases a reviewer should challenge.

In [11]:
if top10.empty:
    print("No rows passed the fixed rule, so there are no top-10 weak picks to list.")
else:
    weak_picks = top10.sort_values(
        ["early_impressions", "early_clicks", "baseline_action_score"],
        ascending=[True, True, True],
    ).head(min(3, len(top10)))
    for _, row in weak_picks.iterrows():
        print(
            f"{int(row['baseline_rank'])}. {row['content_hash_id']}: "
            f"weak-pick concern — only {row['early_impressions']:,.0f} early impressions "
            f"and {row['early_clicks']:,.0f} early clicks; "
            f"the observed decline could be sampling noise or a query-mix change."
        )

10. content_acbcc847f8996314: weak-pick concern — only 83,715 early impressions and 133 early clicks; the observed decline could be sampling noise or a query-mix change.
9. content_3df3f32f3fd58dea: weak-pick concern — only 84,041 early impressions and 120 early clicks; the observed decline could be sampling noise or a query-mix change.
8. content_7c6373141eae744a: weak-pick concern — only 86,860 early impressions and 51 early clicks; the observed decline could be sampling noise or a query-mix change.


## 5) Self-check

The checks below guard against accidental target leakage, product-decision leakage, multiple reason codes, and a queue that was not actually written. The CSV is intentionally generated locally and is not committed to Git; the notebook regenerates it on every run.

In [13]:
assert duplicate_check == 0
assert pages["early_observations"].ge(10).all()
assert pages["late_observations"].ge(10).all()
assert set(queue["reason_code"].unique()) == {"visible_and_due_for_review"}
assert set(queue["action_label"].unique()) == {"review_search_performance"}
assert np.isfinite(queue["baseline_action_score"]).all()
assert (queue["baseline_action_score"] >= 0).all()
assert OUTPUT_CSV.exists()
assert pd.read_csv(OUTPUT_CSV).shape[0] == len(queue)

# Verify the score is reproducible from early-window inputs only.
# Verify the score uses early-period impressions only.
expected_score = np.log1p(queue["early_impressions"])

assert np.allclose(
    queue["baseline_action_score"],
    expected_score
), "Baseline score does not match the early-impressions formula."

metrics = {
    "lane": "Content-Day Performance / Decline Risk",
    "prediction_cutoff": "2026-03-16",
    "feature_window": {"start": EARLY_START, "end": EARLY_END},
    "outcome_window": {"start": LATE_START, "end": LATE_END},
    "eligible_rows": int(len(queue)),
    "decline_proxy_base_rate": base_rate,
    "baseline_precision_at_k": baseline_metrics,
    "baseline_score": "log1p(early_impressions)",
    "score_inputs": ["early_impressions"],
    "label": "late clicks per measured day <= 0.8 * early clicks per measured day",
    "output_csv": str(OUTPUT_CSV.relative_to(REPO_ROOT)),
    "note": "Late-window fields define the outcome only; they do not enter the ranking score."
}
METRICS_JSON = OUTPUT_DIR / "baseline_metrics.json"
METRICS_JSON.write_text(json.dumps(metrics, indent=2))
print("SELF-CHECK PASSED")
print("Metrics receipt:", METRICS_JSON)
print("CSV regenerated:", OUTPUT_CSV)
print("Commit this notebook and baseline_metrics.json; keep baseline_action_score.csv out of Git.")

SELF-CHECK PASSED
Metrics receipt: /content/FlyRank_Internship/work/outputs/baseline_metrics.json
CSV regenerated: /content/FlyRank_Internship/work/outputs/baseline_action_score.csv
Commit this notebook and baseline_metrics.json; keep baseline_action_score.csv out of Git.
